In [73]:
# Imports

import numpy as np
import sympy as sp
from sympy import symbols, Function, diff, tanh, sinh, exp, sqrt, simplify
from sympy.utilities.lambdify import lambdify
from scipy.integrate import solve_ivp
from pyswarms.single import GlobalBestPSO
import matplotlib.pyplot as plt
from sympy.core.relational import Relational



In [74]:
# Symbolic variables and parameters

t = sp.symbols('t')

# Number of parameters to be identified (Input the number of parameters to be identified)
n_vars = 8
kk = sp.symbols('kk1:%d' % (n_vars+1))  # k1, k2, ..., k8
# kk = sp.symbols('k1:9')  # k1 to k8

print(kk)

# C-rate (Put your C-rate)
Crate = -1

# Experimental data (Users can input their experimental conditions
Numexp = 63
Totexp = 3100

# Node number (Change # of node- N: Cathode, M: Membrance, NM: Cathode) (Put your number of node points)
N, M, NM = 2, 2, 2

# Design parameters
ep, es, en = 0.335, 0.47, 0.25
brugp, brugs, brugn = 2.43, 2.57, 2.91
lp, ls, ln1 = 75.6e-6, 12e-6, 85.2e-6
Rpp, Rpn = 5.22e-6, 5.86e-6
F = 96487
R = 8.3143
t1 = 0.363
ap = (3/Rpp)*(1-ep)
an = (3/Rpn)*(1-en)
T = 298.15
Acell = 0.11
Capa = 5
iapp = Capa * Crate / Acell

# Transport params symbolic with kk
c0 = 1000
D1 = kk[0] * 1e-9
Kappa = kk[1]
ctp = 51765
ctn = 29583
Dbulk = D1
sigmap = kk[2]
sigman = kk[3]
Dsp = kk[4] * 1e-15
Dsn = kk[5] * 1e-14

Keffp = Kappa * (ep ** brugp)
Keffs = Kappa * (es ** brugs)
Keffn = Kappa * (en ** brugn)
D2pos = (ep ** brugp) * Dbulk
D2sep = (es ** brugs) * Dbulk
D2neg = (en ** brugn) * Dbulk

kp = kk[6] * 1e-11
kn = kk[7] * 1e-12

h = lp/(N+1)
h2 = ls/(M+1)
h3 = ln1/(NM+1)

print(f"h  = {h}")
print(f"h2 = {h2}")
print(f"h3 = {h3}")


# Symbolic state variables X_i(t)
Nt = 1 + N + 1 + M + 1 + NM + 1 + N + NM + N + NM + N + 2 + NM + 2 + 1 + N + 1 + M + 1 + NM + 1

X = [Function(f'X{i+1}')(t) for i in range(Nt)]

(kk1, kk2, kk3, kk4, kk5, kk6, kk7, kk8)
h  = 2.52e-05
h2 = 4e-06
h3 = 2.84e-05


In [75]:
print(X)

[X1(t), X2(t), X3(t), X4(t), X5(t), X6(t), X7(t), X8(t), X9(t), X10(t), X11(t), X12(t), X13(t), X14(t), X15(t), X16(t), X17(t), X18(t), X19(t), X20(t), X21(t), X22(t), X23(t), X24(t), X25(t), X26(t), X27(t), X28(t), X29(t), X30(t), X31(t), X32(t), X33(t), X34(t), X35(t), X36(t)]


In [76]:
# Create u1, u2, u3, u4, u5 arrays (mapping symbolic vars)


# Electrolyte concentration u1 (length = 1+N+1+M+1+NM+1)
u1_len = 1+N+1+M+1+NM+1
u1 = X[0:u1_len]

print('u1:', u1)

# Surface concentration u2 (length = N + NM)
u2 = [sp.sympify(0)] * 9

# First loop: i = 0 to 1 (N=2)
for i in range(N):
    index = i + (N + 1) + (M + 1) + (NM + 1)  # = i + 10
    u2[i + 1] = X[index + 1]  # u2[1] = X[11], u2[2] = X[12]

# Second loop: i = 0 to 1 (NM=2)
for i in range(NM):
    index = i + (N + 1) + (M + 1) + (NM + 1) + N  # = i + 12
    u2[i + 1 + N + 1 + M + 1] = X[index + 1]  # u2[7] = X[13], u2[8] = X[14]

print('u2:', u2)

# Average concentration u3 (length = N + NM)
u3 = [sp.sympify(0)] * 9
for i in range(N):
    u3[i + 1] = X[i + (N+1) + (M+1) + (NM+1) + N + NM + 1]

for i in range(1, NM+1):
    u3[i + 1 + N + 1 + M] = X[i + 1 + (N+1) + (NM+1) + N + NM + N + 2]

print('u3:', u3)

# Solid phase potential u4 (length = N + 2 + NM + 2)
u4 = [sp.sympify(0)] * 10
for i in range(N+2):
    u4[i] = X[i + (N+1) + (M+1) + (NM+1) + N + NM + N + NM + 1]

for i in range(NM+2):
    u4[i + 1 + N + M + 1] = X[i + (N+1) + (M+1) + (NM+1) + N + NM + N + NM + N + N + 1]

print('u4:', u4)

# Liquid potential u5 (length = 1 + N + 1 + M + 1 + NM + 1)
u5 = [sp.sympify(0)] * 10
for i in range(10):
    u5[i] = X[i+1+N+1+M+1+NM+1+N+NM+N+NM+N+2+NM+2]
    
print('u5:', u5)    
    


u1: [X1(t), X2(t), X3(t), X4(t), X5(t), X6(t), X7(t), X8(t), X9(t), X10(t)]
u2: [0, X11(t), X12(t), 0, 0, 0, 0, X13(t), X14(t)]
u3: [0, X15(t), X16(t), 0, 0, 0, 0, X17(t), X18(t)]
u4: [X19(t), X20(t), X21(t), X22(t), 0, 0, X23(t), X24(t), X25(t), X26(t)]
u5: [X27(t), X28(t), X29(t), X30(t), X31(t), X32(t), X33(t), X34(t), X35(t), X36(t)]


In [77]:
# Compute jp (molar flux)
# at positive electrode
jp = [sp.simplify(0)]*(N+1)
# jp = [sp.sympify(0)] * 10

for i in range(1, N+1):
    theta = u2[i]*ctp/ctp  # theta = u2(i)*ctp/ctp = u2(i)
    Up = (-0.8090)*theta + 4.4875 - 0.0428*tanh(18.5138*(theta-0.5542)) - 17.7326*tanh(15.7890*(theta-0.3117)) + 17.5842*tanh(15.9308*(theta-0.3120))
    jp[i] = 2*kp*sqrt(u1[i]*c0)*sqrt(ctp - u2[i]*ctp)*sqrt(u2[i]*ctp)*sinh(0.5*F/(R*T)*(u4[i] - u5[i] - Up))
    
# print('jp:', jp) 
print('len(jp):', len(jp))
    
# Display the results
print("jp = [")
for expr in jp:
    print(expr)
print("]")

len(jp): 3
jp = [
0
1.0e-9*sqrt(20706)*kk7*sqrt(51765 - 51765*X11(t))*sqrt(X11(t))*sqrt(X2(t))*sinh(15.7444257387882*X11(t) + 19.4615892939286*X20(t) - 19.4615892939286*X28(t) + 345.104578313517*tanh(15.789*X11(t) - 4.9214313) - 342.216478462298*tanh(15.9308*X11(t) - 4.9704096) + 0.832956021780142*tanh(18.5138*X11(t) - 10.26034796) - 87.3338819565044)
1.0e-9*sqrt(20706)*kk7*sqrt(51765 - 51765*X12(t))*sqrt(X12(t))*sqrt(X3(t))*sinh(15.7444257387882*X12(t) + 19.4615892939286*X21(t) - 19.4615892939286*X29(t) + 345.104578313517*tanh(15.789*X12(t) - 4.9214313) - 342.216478462298*tanh(15.9308*X12(t) - 4.9704096) + 0.832956021780142*tanh(18.5138*X12(t) - 10.26034796) - 87.3338819565044)
]


In [78]:
    
# at negative electrode
jn = [sp.simplify(0)]* (1 + N + 1 + M + 1 + NM)

# Loop indices
start_idx = 7
end_idx = 9

print(f"Length of jn: {len(jn)}")

for i in range(start_idx, end_idx):
    theta = u2[i]*ctn/ctn
    
    # Un = ((1.9793) * sp.exp(-39.3631 * theta) +
    #       0.2482 -
    #       0.0909 * sp.tanh(29.8538 * (theta - 0.1234)) -
    #       0.04478 * sp.tanh(14.9159 * (theta - 0.2769)) -
    #       0.0205 * sp.tanh(30.4444 * (theta - 0.6103)))
    
    Un = (0.2482
      + 1.9793 * theta * sp.exp(-39.3631 * theta)
      + 0.0909 * sp.tanh(29.8538 * (theta - 0.1234))
      - 0.04478 * sp.tanh(14.9159 * (theta - 0.2769))
      - 0.0205 * sp.tanh(30.4444 * (theta - 0.2769 - 0.6103)))
    
    jn[i] = 2*kn*sqrt(u1[i]*c0)*sqrt(ctn - u2[i]*ctn)*sqrt(u2[i]*ctn)*sinh(0.5*F/(R*T)*(u4[i] - u5[i] - Un))


# Display the results
print("jn = [")
for expr in jn:
    print(expr)
print("]")


Length of jn: 9
jn = [
0
0
0
0
0
0
0
-6.0e-11*sqrt(32870)*kk8*sqrt(29583 - 29583*X13(t))*sqrt(X13(t))*sqrt(X8(t))*sinh(38.5203236894728*X13(t)*exp(-39.3631*X13(t)) - 19.4615892939286*X24(t) + 19.4615892939286*X34(t) - 0.87148996858212*tanh(14.9159*X13(t) - 4.13021271) + 1.76905846681811*tanh(29.8538*X13(t) - 3.68395892) - 0.398962580525535*tanh(30.4444*X13(t) - 27.01027168) + 4.83036646275307)
-6.0e-11*sqrt(32870)*kk8*sqrt(29583 - 29583*X14(t))*sqrt(X14(t))*sqrt(X9(t))*sinh(38.5203236894728*X14(t)*exp(-39.3631*X14(t)) - 19.4615892939286*X25(t) + 19.4615892939286*X35(t) - 0.87148996858212*tanh(14.9159*X14(t) - 4.13021271) + 1.76905846681811*tanh(29.8538*X14(t) - 3.68395892) - 0.398962580525535*tanh(30.4444*X14(t) - 27.01027168) + 4.83036646275307)
]


In [79]:
#  Form PDE/ODE equations (electrolyte concentration in positive electrode)
#u1: Electrolyte concentration (mol/m3)

# finite difference spatial derivatives approximations
dudxf1 = 1/(2*h) * (-u1[2] - 3*u1[0] + 4*u1[1])
dudxb1 = 1/(2*h) * (u1[N-1] + 3*u1[N+1] - 4*u1[N])
dudxf1_2 = 1/(2*h2) * (-u1[N+3] - 3*u1[N+1] + 4*u1[N+2])

bc11 = dudxf1
bc21 = D2pos * dudxb1 - D2sep * dudxf1_2


# Initialize eq1 list
eq1 = [sp.simplify(0)]* (1 + N + 1 + M + 1 + NM + 1)
print(eq1)
# eq1 = sp.zeros(1, 1 + N + 1 + M + 1 + NM + 1)

# eq1[0] = 0 - bc11
eq1[0] = sp.Eq(0, bc11)


for i in range(1, N):
    d2udx21 = (1 / h ** 2) * (u1[i-2] - 2 * u1[i-1] + u1[i])
    eq1[i-1] = sp.Eq(diff(u1[i-1]), (D2pos * d2udx21 + ap * (1 - t1) * jp[i-1] / c0) / ep)

# eq1[N + 1] = 0 - bc21

eq1[N + 1] = sp.Eq(0, bc21)

# Separator
dudxb1_2 = (1 / (2 * h2)) * (u1[N + M] + 3 * u1[N + M + 2] - 4 * u1[N + M + 1])
dudxf1_3 = (1 / (2 * h3)) * (-u1[N + M + 4] - 3 * u1[N + M + 2] + 4 * u1[N + M + 3])
bc31 = D2sep * dudxb1_2 - D2neg * dudxf1_3

for i in range(N + 2, N + M + 1):
    d2udx21 = (1 / h2 ** 2) * (u1[i - 2] - 2 * u1[i-1] + u1[i])
    eq1[i-1] = sp.Eq(diff(u1[i-1]), D2sep * d2udx21 / es)

# eq1[N + M + 2] = 0 - bc31
eq1[N + M + 2] = sp.Eq(0, bc31)

# Negative Electrode
dudxb1_3 = (1 / (2 * h3)) * (u1[1+N+1+M+1+NM-2] + 3 * u1[1+N+1+M+1+NM] - 4 * u1[1+N+1+M+1+NM-1])
bc41 = dudxb1_3

for i in range(N + M + 3, N + M + NM + 2):
    d2udx21 = (1 / h3 ** 2) * (u1[i - 2] - 2 * u1[i - 1] + u1[i])
    eq1[i-1] = sp.Eq(diff(u1[i-1]), (D2neg * d2udx21 + an * (1 - t1) * jn[i-1] / c0) / en)

# eq1[1+N+1+M+1+NM] = 0 - bc41
eq1[1+N+1+M+1+NM] = sp.Eq(0, bc41)

# u2: Surface concentration
eq2 = [sp.simplify(0)]* (N + M + NM + 3)

print(eq2)

# Positive electrode
for i in range(1, N):
    eq2[i-1] = sp.Eq(0,-u2[i-1] + u3[i-1] - jp[i-1] * Rpp / Dsp / 5 / ctp)

# Negative electrode
for i in range(N+2+M+1,N+2+M+1+NM-1):
    eq2[i-1] = sp.Eq(0,-u2[i-1] + u3[i-1] - jn[i-1] * Rpn / Dsn / 5 / ctn)

# u3: Average concentration
eq3 = [sp.simplify(0)]* (1+N+1+M+1+NM)
print(eq3)
for i in range(1, N):
    eq3[i-1] = sp.Eq(diff(u3[i-1]), - 3 * jp[i-1] / Rpp / ctp)

for i in range(1+N+1+M+1, 1+N+1+M+1+NM-1):
    eq3[i-1] = sp.Eq(diff(u3[i-1]), - 3 * jn[i-1] / Rpn / ctn)

# u4: Solid potential
eq4 = [sp.simplify(0)]* (1+N+1+M+1+NM+1)
print(eq4)

dudxf4 = (1 / (2 * h)) * (-u4[2] - 3 * u4[0] + 4 * u4[1])
dudxb4 = (1 / (2 * h)) * (u4[N - 1] + 3 * u4[N + 1] - 4 * u4[N])

bc14 = dudxf4 + iapp / sigmap
bc24 = dudxb4

# eq4[0] = 0 - bc14
eq4[0] = sp.Eq(0, bc14)

for i in range(1, N):
    d2udx24 = (1 / h ** 2) * (u4[i - 2] - 2 * u4[i-1] + u4[i])
    eq4[i-1] = sp.Eq(0,d2udx24 - ap * F * jp[i] / sigmap)

# eq4[N + 1] = 0 - bc24
eq4[N + 1] = sp.Eq(0, bc24)

# Negative
dudxf4_3 = (1 / (2 * h3)) * (-u4[N+2+M+1+1] - 3 * u4[N + M + 2] + 4 * u4[N + M + 3])
dudxb4_3 = (1 / (2 * h3)) * (u4[N + M + NM + 1] + 3 * u4[N + M + NM + 3] - 4 * u4[N + M + NM + 2])

bc34 = dudxf4_3
bc44 = dudxb4_3 + iapp / sigman

# eq4[N + M + 2] = 0 - bc34
eq4[N + M + 2] = sp.Eq(0, bc34)

for i in range(N+2+M+1, N + M + NM + 2):
    d2udx24 = (1 / h3 ** 2) * (u4[i - 2] - 2 * u4[i-1] + u4[i])
    eq4[i-1] = sp.Eq(0,d2udx24 - an * F * jn[i-1] / sigman)

# eq4[N + M + NM + 2] = 0 - bc44
eq4[1+N+1+M+1+NM] = sp.Eq(0, bc44)

# u5: Liquid phase potential (V)
# eq5 = sp.zeros(1, N + M + NM + 4)
eq5 = [sp.simplify(0)]* (1+N+1+M+1+NM+1)
print(eq5)

dudxf5 = (1 / (2 * h)) * (-u5[2] - 3 * u5[0] + 4 * u5[1])
dudxb5 = (1 / (2 * h)) * (u5[N - 1] + 3 * u5[N + 1] - 4 * u5[N])
dudxf5_2 = (1 / (2 * h2)) * (-u5[N + 3] - 3 * u5[N +1] + 4 * u5[N+2])

bc15 = dudxf5
bc25 = Keffp * dudxb5 - Keffs * dudxf5_2

# eq5[0] = 0 - bc15
eq5[0] = sp.Eq(0, bc15)

for i in range(1, N):
    dudx1 = (1 / (2 * h)) * (u1[i] - u1[i - 2])
    dudx4 = (1 / (2 * h)) * (u4[i] - u4[i - 2])
    dudx5 = (1 / (2 * h)) * (u5[i] - u5[i - 2])
    eq5[i-1] = sp.Eq(0,-sigmap * dudx4 - Keffp * dudx5 + (2 * Keffp * R * T * (1 - t1) * dudx1) / (F * u1[i-1]) - iapp)
    
eq5[N + 1] = sp.Eq(0, bc25)

# Separator
for i in range(N + 2, N + 1 + M):
    dudx1 = (1 / (2 * h2)) * (u1[i] - u1[i-2])
    dudx5 = (1 / (2 * h2)) * (u5[i] - u5[i-2])
    eq5[i - 1] = sp.Eq(0, -Keffs * dudx5 + (2 * Keffs * R * T * (1 - t1) * dudx1) / (F * u1[i-1]) - iapp)


dudxb5_2 = (1 / (2 * h2)) * (u5[N + M ] + 3 * u5[N + M + 2] - 4 * u5[N + M + 1])
dudxf5_3 = (1 / (2 * h3)) * (-u5[N + M + 4] - 3 * u5[N + M + 2] + 4 * u5[N + M + 3])
bc35 = Keffs * dudxb5_2 - Keffn * dudxf5_3

eq5[N + M + 2] = sp.Eq(0, bc35)

for i in range(N + M + 3, N + M + NM + 2):
    dudx1 = (1 / (2 * h3)) * (u1[i] - u1[i-2])
    dudx4 = (1 / (2 * h3)) * (u4[i] - u4[i-2])
    dudx5 = (1 / (2 * h3)) * (u5[i] - u5[i-2])
    eq5[i-1] = sp.Eq(0, -sigman * dudx4 - Keffn * dudx5 + (2 * Keffn * R * T * (1 - t1) * dudx1) / (F * u1[i-1]) - iapp)


bc45 = u5[N + M + NM + 3]
eq5[N + M + NM + 3] = sp.Eq(0, bc45)




[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## A single-step iteration-free initialization approach 

In [ ]:
mu = 10**(-3)
q = 1000
initime = 200
ff = 1/2 * tanh(q * (t - initime)) + 1/2

length1 = 1 + N + 1 + M + 1 + NM + 1 
eqn1 = [sp.simplify(0)]* length1

eqn1[0] = -mu * (diff(eq1[0].rhs, t) - diff(eq1[0].lhs, t)) - eq1[0].rhs + eq1[0].lhs

for ii in range(1, N):
    eqn1[ii-1] = eq1[ii-1].lhs - eq1[ii-1].rhs * ff

eqn1[N+1] = -mu * (diff(eq1[N+1].rhs, t) - diff(eq1[N+1].lhs, t)) - eq1[N+1].rhs + eq1[N+1].lhs

for ii in range(N+2, N+2+M):
    eqn1[ii] = eq1[ii].lhs - eq1[ii].rhs * ff

eqn1[N+2+M] = -mu * (diff(eq1[N+2+M].rhs, t) - diff(eq1[N+2+M].lhs, t)) - eq1[N+2+M].rhs + eq1[N+2+M].lhs

for ii in range(N+2+M+1, N+2+M+1+NM):
    eqn1[ii] = eq1[ii].lhs - eq1[ii].rhs * ff

eqn1[length1 - 1] = -mu * (diff(eq1[length1 - 1].rhs, t) - diff(eq1[length1 - 1].lhs, t)) - eq1[length1 - 1].rhs + eq1[length1 - 1].lhs

# eqn2
eqn2 = [0] * (N + 2 + M + 1 + NM)
for ii in range(1, N+1):
    eqn2[ii] = -mu * (diff(eq2[ii].rhs, t) - diff(eq2[ii].lhs, t)) - eq2[ii].rhs + eq2[ii].lhs

for ii in range(N+2+M, N+2+M+NM):
    eqn2[ii] = -mu * (diff(eq2[ii].rhs, t) - diff(eq2[ii].lhs, t)) - eq2[ii].rhs + eq2[ii].lhs

# eqn3
eqn3 = [0] * (1 + N + 1 + M + 1 + NM)
for ii in range(1, N+1):
    eqn3[ii] = eq3[ii].lhs - eq3[ii].rhs * ff

for ii in range(1 + N + 1 + M + 1, 1 + N + 1 + M + 1 + NM):
    eqn3[ii] = eq3[ii].lhs - eq3[ii].rhs * ff

# eqn4
eqn4 = [0] * (1 + N + 1 + M + 1 + NM + 1)
eqn4[0] = -mu * (diff(eq4[0].rhs, t) - diff(eq4[0].lhs, t)) - eq4[0].rhs + eq4[0].lhs

for ii in range(1, N+1):
    eqn4[ii] = -mu * (diff(eq4[ii].rhs, t) - diff(eq4[ii].lhs, t)) - eq4[ii].rhs + eq4[ii].lhs

eqn4[N+1] = -mu * (diff(eq4[N+1].rhs, t) - diff(eq4[N+1].lhs, t)) - eq4[N+1].rhs + eq4[N+1].lhs
eqn4[N+2+M] = -mu * (diff(eq4[N+2+M].rhs, t) - diff(eq4[N+2+M].lhs, t)) - eq4[N+2+M].rhs + eq4[N+2+M].lhs

for ii in range(N+2+M+1, N+2+M+1+NM):
    eqn4[ii] = -mu * (diff(eq4[ii].rhs, t) - diff(eq4[ii].lhs, t)) - eq4[ii].rhs + eq4[ii].lhs

eqn4[-1] = -mu * (diff(eq4[-1].rhs, t) - diff(eq4[-1].lhs, t)) - eq4[-1].rhs + eq4[-1].lhs

# eqn5
eqn5 = [0] * (1 + N + 1 + M + 1 + NM + 1)
eqn5[0] = -mu * (diff(eq5[0].rhs, t) - diff(eq5[0].lhs, t)) - eq5[0].rhs + eq5[0].lhs

for ii in range(1, N+1):
    eqn5[ii] = -mu * (diff(eq5[ii].rhs, t) - diff(eq5[ii].lhs, t)) - eq5[ii].rhs + eq5[ii].lhs

eqn5[N+1] = -mu * (diff(eq5[N+1].rhs, t) - diff(eq5[N+1].lhs, t)) - eq5[N+1].rhs + eq5[N+1].lhs

for ii in range(N+2, N+2+M):
    eqn5[ii] = -mu * (diff(eq5[ii].rhs, t) - diff(eq5[ii].lhs, t)) - eq5[ii].rhs + eq5[ii].lhs

eqn5[N+2+M] = -mu * (diff(eq5[N+2+M].rhs, t) - diff(eq5[N+2+M].lhs, t)) - eq5[N+2+M].rhs + eq5[N+2+M].lhs

for ii in range(N+2+M+1, N+2+M+1+NM):
    eqn5[ii] = -mu * (diff(eq5[ii].rhs, t) - diff(eq5[ii].lhs, t)) - eq5[ii].rhs + eq5[ii].lhs

eqn5[-1] = -mu * (diff(eq5[-1].rhs, t) - diff(eq5[-1].lhs, t)) - eq5[-1].rhs + eq5[-1].lhs


In [ ]:
# Define symbolic constants
mu = 10**(-3)
q = 1000
initime = 200
ff = 1/2 * tanh(q * (t - initime)) + 1/2

# Assumed previously defined variables:
# eq1, eq2, eq3, eq4, eq5 are dicts/lists of Eq() objects

# Create symbolic arrays for modified equations
size_full = 1 + N + 1 + M + 1 + NM + 1
size_part = N + 2 + M + 1 + NM

# eq1
for i in range(size_full):
    if i == 0 or i == N + 1 or i == N + M + 2 or i == size_full - 1:
        eq1[i] = -mu * (diff(rhs(eq1[i]), t) - diff(lhs(eq1[i]), t)) - rhs(eq1[i]) + lhs(eq1[i])
    else:
        eq1[i] = lhs(eq1[i]) - rhs(eq1[i]) * ff

# eq2
for i in range(size_part):
    if 2 <= i <= N + 1 or N + M + 2 <= i < size_part:
        eq2[i] = -mu * (diff(rhs(eq2[i]), t) - diff(lhs(eq2[i]), t)) - rhs(eq2[i]) + lhs(eq2[i])

# eq3
for i in range(size_full - 1):
    if 2 <= i <= N + 1 or N + M + 3 <= i < size_full:
        eq3[i] = lhs(eq3[i]) - rhs(eq3[i]) * ff

# eq4
for i in range(size_full):
    if i in [0, N + 1, N + M + 2, size_full - 1] or 2 <= i <= N + 1 or N + M + 3 <= i < size_full - 1:
        eq4[i] = -mu * (diff(rhs(eq4[i]), t) - diff(lhs(eq4[i]), t)) - rhs(eq4[i]) + lhs(eq4[i])

# eq5
for i in range(size_full):
    if i in [0, N + 1, N + M + 2, size_full - 1] or 2 <= i <= N + 1 or N + 2 <= i <= N + M + 1 or N + M + 3 <= i < size_full - 1:
        eq5[i] = -mu * (diff(rhs(eq5[i]), t) - diff(lhs(eq5[i]), t)) - rhs(eq5[i]) + lhs(eq5[i])


In [ ]:
# Python version of the MATLAB execution, solving, and optimization pipeline for the P2D model

import numpy as np
from sympy import Matrix, symbols, diag, lambdify
from scipy.integrate import solve_ivp
from scipy.optimize import differential_evolution
import matplotlib.pyplot as plt

# Placeholders for your symbolic variables (to be replaced by actual SymPy symbolic equations and variables)
eqn1, eqn2, eqn3, eqn4, eqn5 = [None]*5  # To be replaced with actual symbolic expressions
varsX = symbols('x0:100')  # Placeholder: adjust range and naming according to your model
kk = symbols('k0:8')

# Combine equations
eqs = eqn1 + eqn2[1:N+1] + eqn2[N+2+M+1:] + eqn3[1:N+1] + eqn3[N+2+M+1:] + eqn4[:N+2] + eqn4[1+N+1+M:] + eqn5

# Mass matrix formulation
MM_sym, f_sym = Matrix(eqs).as_explicit()

# Convert symbolic expressions to numerical functions
MM_func = lambdify((varsX, kk), MM_sym, modules='numpy')
f_func = lambdify((varsX, kk), f_sym, modules='numpy')

t = symbols('t')
mu = 1e-3
q = 1000
initime = 200
ff = 0.5 * np.tanh(q*(t - initime)) + 0.5

# Initial guess (adjust sizes according to actual model size)
U = np.zeros(1 + N + 1 + M + 1 + NM + 1 + 1 + N + 1 + M + 1 + NM + 1 + N + NM + N + NM + N + 2 + NM + 2)
U[:1+N+1+M+1+NM+1] = 1
U[1+1+N+1+M+1+NM+1:N+1+N+1+M+1+NM+1] = 0.27
U[1+1+N+1+M+1+NM+1+N:NM+1+N+1+M+1+NM+1+N] = 0.9014
U[1+1+N+1+M+1+NM+1+N+NM:N+1+N+1+M+1+NM+1+N+NM] = 0.27
U[1+1+N+1+M+1+NM+1+N+NM+N:NM+1+N+1+M+1+NM+1+N+NM+N] = 0.9014
U[1+1+N+1+M+1+NM+1+N+NM+N+NM:N+2+1+N+1+M+1+NM+1+N+NM+N+NM] = 4.30430037
U[1+1+N+1+M+1+NM+1+N+NM+N+NM+N+2:NM+2+1+N+1+M+1+NM+1+N+NM+N+NM+N+2] = 0.09202000152
U[1+1+N+1+M+1+NM+1+N+NM+N+NM+N+2+NM+2:] = 0

y0 = U.copy()

# Experimental data
x_exp = np.loadtxt('voltage_exp.txt')

# PSO bounds and parameters
pp = 0.3
init_params = [1, 1.17, 0.18, 215, 4, 3.3, 0.7, 0.7]
lower_bounds = [(1-pp)*p for p in init_params]
upper_bounds = [(1+pp)*p for p in init_params]

# Objective function
def P2Dobj(kk_):
    try:
        M0 = MM_func(y0, kk_)
        vw = 1 / np.maximum(np.abs(M0).max(axis=1), 1e-10)
        mw = np.diag(vw)

        def F(t, y):
            return vw * f_func(y, kk_)

        def M1(t, y):
            return mw @ MM_func(y, kk_)

        t_span = (0, Totexp + 200)
        t_eval = np.linspace(*t_span, Numexp + 3)

        sol = solve_ivp(F, t_span, y0, method='BDF', t_eval=t_eval, atol=1e-5, rtol=1e-5)
        V_model = sol.y[var1_index] - sol.y[var2_index]  # Fill with correct indices
        return np.sqrt(np.mean((x_exp[:, 0] - V_model[3:])**2))
    except:
        return 1000

# Run PSO (replaced with differential evolution for Python)
result = differential_evolution(P2Dobj, bounds=list(zip(lower_bounds, upper_bounds)), strategy='best1bin',
                                maxiter=10, popsize=10, disp=True)
kk_opt = result.x

# Extracted parameters
D1, Kappa, sigmap, sigman, Dsp, Dsn, kp, kn = kk_opt
D1 *= 1e-9
Dsp *= 1e-15
Dsn *= 1e-14
kp *= 1e-11
kn *= 1e-12

# Final solve using optimal params
M0 = MM_func(y0, kk_opt)
vw = 1 / np.maximum(np.abs(M0).max(axis=1), 1e-10)
mw = np.diag(vw)

F = lambda t, y: vw * f_func(y, kk_opt)
M1 = lambda t, y: mw @ MM_func(y, kk_opt)

sol = solve_ivp(F, (0, 100000), y0, method='BDF', atol=1e-5, rtol=1e-5)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(sol.t - initime, sol.y[var1_index] - sol.y[var2_index], label='P2D Model', linewidth=2)
plt.plot(np.linspace(0, Totexp, Numexp), x_exp[:, 0], 'ro', label='Experiment')
plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')
plt.legend()
plt.grid(True)
plt.savefig('voltage_25C.bmp')
plt.show()


In [ ]:
# Numeric solution preparation (Mass matrix and ODE RHS functions)

# Given symbolic variables
t = sp.symbols('t')
kk = sp.symbols('kk0:8')  # kk0 to kk7, 8 parameters

# Assuming 'eqs' is a list of symbolic equations with lhs - rhs = 0 form
# And 'vars' is list of dependent variables: X_i(t)

# Rearrange eqs to get M * y_dot = f
# We separate time derivatives and algebraic parts
# Extract coefficients for derivatives and build M matrix and f vector

def mass_matrix_form(eqs, vars, t):
    """
    Given symbolic eqs of the form lhs == rhs,
    separate terms into M*y_dot = f
    Returns symbolic M matrix and f vector.
    """
    n = len(vars)
    M = sp.zeros(n)
    f_vec = sp.zeros(n, 1)

    for i, eq in enumerate(eqs):
        # lhs - rhs = 0  => eq = 0
        # Isolate derivatives (diff(vars[i], t)) terms
        # Collect coefficients
        eq = sp.simplify(eq)
        deriv = sp.Derivative(vars[i], t)
        coeff = eq.coeff(deriv)
        
        if coeff != 0:
            # Put coeff in M[i,i]
            M[i,i] = coeff
            # f_i = eq without derivative term
            f_vec[i] = eq - coeff * deriv
        else:
            # Algebraic eq (no derivative)
            M[i,i] = 0
            f_vec[i] = eq

    return M, f_vec

# Example usage
# M_sym, f_sym = mass_matrix_form(eqs, varsX, t)

# Now we convert M and f to numerical functions
# We must replace symbolic kk and vars with numerical arrays

from sympy.utilities.lambdify import lambdify

def generate_numeric_functions(M_sym, f_sym, varsX, kk):
    """
    Create callable numerical functions for M(t,y,kk) and f(t,y,kk)
    """
    # Flatten varsX and kk into list of symbols for lambdify
    variables = [t] + varsX + list(kk)
    
    # Lambdify M matrix and f vector element-wise (for efficiency)
    M_func = sp.lambdify(variables, M_sym, modules='numpy')
    f_func = sp.lambdify(variables, f_sym, modules='numpy')

    def M_numeric(time, y, params):
        args = [time] + list(y) + list(params)
        return np.array(M_func(*args), dtype=float)

    def f_numeric(time, y, params):
        args = [time] + list(y) + list(params)
        return np.array(f_func(*args), dtype=float).flatten()

    return M_numeric, f_numeric

# Then you would do:
# M_numeric, f_numeric = generate_numeric_functions(M_sym, f_sym, varsX, kk)


In [ ]:
U(1:1+N+1+M+1+NM+1) = 1;  % Electrolyte concentration
U(1+1+N+1+M+1+NM+1:N+1+N+1+M+1+NM+1) = 0.27;  % Surface concentration at positive
U(...) = ... % and so on


In [ ]:
# Parameter bounds for PSO (Particle Swarm Optimization)

# Define bounds arrays based on your MATLAB variables

pp = 0.3
D10 = 1
Kappa0 = 1.17
sigmap0 = 0.18
sigman0 = 215
Dsp0 = 4
Dsn0 = 3.3
kp0 = 0.7
kn0 = 0.7

lower_bound = np.array([D10*(1-pp), Kappa0*(1-pp), sigmap0*(1-pp), sigman0*(1-pp), Dsp0*(1-pp), Dsn0*(1-pp), kp0*(1-pp), kn0*(1-pp)])
upper_bound = np.array([D10*(1+pp), Kappa0*(1+pp), sigmap0*(1+pp), sigman0*(1+pp), Dsp0*(1+pp), Dsn0*(1+pp), kp0*(1+pp), kn0*(1+pp)])

bounds = (lower_bound, upper_bound)

# Define your objective function like P2Dobj in Python to run PSO
def objective_function(kk):
    # Run your ODE solver here with kk parameters, then calculate error (rms)
    # Return error
    pass

# Initialize optimizer
options = {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
optimizer = GlobalBestPSO(n_particles=10, dimensions=8, options=options, bounds=bounds)

best_cost, best_pos = optimizer.optimize(objective_function, iters=10)


In [ ]:
# ODE Solver Setup

def ode_system(t, y, params):
    M = M_numeric(t, y, params)
    f = f_numeric(t, y, params)
    # Solve M * y_dot = f => y_dot = M^{-1} * f
    y_dot = np.linalg.solve(M, f)
    return y_dot

# Define event function like stopcondition

def stop_condition(t, y, params):
    idx1 = 1+N+1+M+1+NM+1+N+NM+N+NM+1
    idx2 = 1+N+1+M+1+NM+1+N+NM+N+NM+N+2+NM+2
    return y[idx1] - y[idx2] - 2.7

stop_condition.terminal = True
stop_condition.direction = 0

# Solve ODE with solve_ivp

t_span = (0, 100000)  # Adjust as needed
y0 = ... # Your initial condition vector
params = best_pos  # Or initial guess

sol = solve_ivp(fun=lambda t,y: ode_system(t,y,params),
                t_span=t_span, y0=y0,
                method='BDF',
                events=lambda t,y: stop_condition(t,y,params),
                rtol=1e-5, atol=1e-5,
                max_step=5)

# sol.t, sol.y contain the solution


In [ ]:


plt.figure()
plt.plot(sol.t - 200, sol.y[idx1, :] - sol.y[idx2, :], linewidth=2, label='P2D Model')
time_exp = np.linspace(0, Totexp, Numexp)
plt.plot(time_exp, voltage_exp, 'o', markersize=7, color='red', label='Experiment')

plt.xlim([0, 3200])
plt.ylim([2.8, 4])
plt.xlabel('Time (seconds)', fontsize=15)
plt.ylabel('Voltage (V)', fontsize=15)
plt.legend()
plt.grid(True)
plt.show()
